# Stage 7 — Scored Differential Diagnosis

LLM produces a **ranked, scored** differential for the **current** admission from:

1. **Symptom tree** (`symptom_tree.json`)
2. **Retained SNOMED context** (`snomed_retained.json`)
3. **Prior PMH (hybrid)** — summarized prior principals + chronic conditions + full ICD list for **most recent** prior stay only
4. **Admission context** from IE (`reason_for_admission`, procedures) — soft guidance only; rank-1 must be a principal **disease**, not a symptom/device event
5. Optional: structured clinical context + full IE (current stay)

**Prompt format:** ROLE / CONTEXT / TASK / CONSTRAINTS  
**Temperature:** `0.4`  
**Does not** use current-stay ground-truth ICD or discharge diagnosis lists.

**Input:** Stage 4–6 exports under `patient_records/`  
**Output:** `data/stage_07_differential_diagnosis/` + per-admission `differential_diagnosis.json` / `.txt`


In [5]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from pipeline import (
    DIFF_DX_CHECKPOINT_JSON,
    DIFF_DX_RESULTS_JSON,
    DIFF_DX_TEMPERATURE,
    EXPORT_DIR,
    LLMNotAvailableError,
    LLM_REQUEST_DELAY_SECONDS,
    STAGE_07_DIR,
    build_prior_icd_context,
    check_llm,
    differential_diagnosis_agent,
    export_diff_dx_to_admission,
    get_llm_config,
    list_admission_export_dirs,
    load_diff_dx_checkpoint,
    print_pipeline_banner,
    save_diff_dx_checkpoint,
    save_diff_dx_results,
    warn_if_slow_model,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")
print(f"DiffDx temperature: {DIFF_DX_TEMPERATURE}")
print(f"Prompt format: ROLE / CONTEXT / TASK / CONSTRAINTS (+ all prior ICDs)")

STAGE_07_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir : {EXPORT_DIR}")
print(f"Stage 7 out: {STAGE_07_DIR}")


Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (qwen/qwen-2.5-7b-instruct, ZDR on)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
OpenRouter ZDR : enabled (provider.zdr=true on every request)
LLM ready — openrouter: qwen/qwen-2.5-7b-instruct
DiffDx temperature: 0.4
Prompt format: ROLE / CONTEXT / TASK / CONSTRAINTS (+ all prior ICDs)
Export dir : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
Stage 7 out: /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_07_differential_diagnosis


In [6]:
admissions = list_admission_export_dirs(EXPORT_DIR)
ready = [a for a in admissions if a["has_symptom_tree"]]
missing_tree = [a for a in admissions if not a["has_symptom_tree"]]
missing_ret = [a for a in ready if not a["has_retained"]]
missing_hist = []
for a in ready:
    hp = EXPORT_DIR / f"patient_{a['patient_id']}" / "admission_history.json"
    if not hp.exists():
        missing_hist.append(a["patient_id"])
print(f"Admissions found     : {len(admissions)}")
print(f"With symptom tree    : {len(ready)}")
print(f"Missing retained SNOMED (run with empty): {len(missing_ret)}")
print(f"Missing admission_history.json: {len(set(missing_hist))}")
if missing_tree:
    print("Skip (no tree):", [(a["patient_id"], a["hadm_id"]) for a in missing_tree[:5]])


Admissions found     : 15
With symptom tree    : 15
Missing retained SNOMED (run with empty): 0
Missing admission_history.json: 0


In [7]:
done = load_diff_dx_checkpoint()
records = list(done.values())
todo = [
    a for a in ready
    if f"{a['patient_id']}|{a['hadm_id']}" not in done
]
print(f"Resuming: {len(done)} done, {len(todo)} remaining")
print("Note: delete diff_dx_checkpoint.json to re-run all with the new prior-ICD prompt.\n")

for i, adm in enumerate(todo, start=1):
    pid, hid = adm["patient_id"], adm["hadm_id"]
    adm_dir = Path(adm["admission_dir"])
    print(f"[{i}/{len(todo)}] DiffDx patient={pid} hadm={hid}...")

    tree = json.loads(Path(adm["symptom_tree_path"]).read_text(encoding="utf-8"))
    retained = {}
    if adm["has_retained"]:
        retained = json.loads(Path(adm["retained_path"]).read_text(encoding="utf-8"))

    # Prior admissions — ALL ICD-10 codes (Option B)
    hist_path = EXPORT_DIR / f"patient_{pid}" / "admission_history.json"
    history = []
    if hist_path.exists():
        history = json.loads(hist_path.read_text(encoding="utf-8"))
        if not isinstance(history, list):
            history = []
    prior_preview = build_prior_icd_context(history)
    n_icds = sum(len(p.get("icd10_diagnoses") or []) for p in prior_preview)
    print(f"  prior admissions={len(prior_preview)} | prior ICD codes={n_icds}")

    ctx_path = adm_dir / "clinical_context.txt"
    clinical_context = ctx_path.read_text(encoding="utf-8") if ctx_path.exists() else None

    ie = None
    ie_path = adm_dir / "information_extraction.json"
    if ie_path.exists():
        ie = json.loads(ie_path.read_text(encoding="utf-8"))

    try:
        result = differential_diagnosis_agent(
            symptom_tree=tree,
            retained_snomed=retained,
            patient_id=pid,
            hadm_id=hid,
            clinical_context_text=clinical_context,
            ie_summary=ie,
            admission_history=history,
            config=LLM_CONFIG,
            temperature=DIFF_DX_TEMPERATURE,
        )
    except (ValueError, TimeoutError, LLMNotAvailableError) as exc:
        print(f"  ERROR: {exc}")
        result = {
            "patient_id": pid,
            "hadm_id": hid,
            "type": "differential_diagnosis",
            "error": str(exc),
            "differential": [],
            "n_candidates": 0,
            "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        }

    export_diff_dx_to_admission(result, adm_dir)
    records.append(result)
    save_diff_dx_checkpoint(records)

    top = (result.get("differential") or [{}])[0]
    print(
        f"  most_likely={result.get('most_likely')!r} | "
        f"n={result.get('n_candidates')} | "
        f"top_score={top.get('score')} | "
        f"temp={result.get('_temperature')}"
    )

    if i < len(todo) and LLM_REQUEST_DELAY_SECONDS > 0:
        time.sleep(LLM_REQUEST_DELAY_SECONDS)

out = save_diff_dx_results(records)
print(f"\nSaved aggregate → {out}")
print(f"Per-admission: {EXPORT_DIR}/patient_*/admissions/hadm_*/differential_diagnosis.*")


Resuming: 13 done, 2 remaining
Note: delete diff_dx_checkpoint.json to re-run all with the new prior-ICD prompt.

[1/2] DiffDx patient=19104262 hadm=24271247...
  prior admissions=5 | prior ICD codes=46
  most_likely='Diabetic ketoacidosis' | n=4 | top_score=85.0 | temp=0.4
[2/2] DiffDx patient=19632936 hadm=26696232...
  prior admissions=1 | prior ICD codes=18
  most_likely='Macrocytic anemia' | n=5 | top_score=85.0 | temp=0.4

Saved aggregate → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_07_differential_diagnosis/differential_diagnoses.json
Per-admission: /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records/patient_*/admissions/hadm_*/differential_diagnosis.*


In [8]:
# Preview first successful run
for row in records:
    if row.get("differential"):
        print(f"Patient {row.get('patient_id')} HADM {row.get('hadm_id')}")
        print(f"Temp={row.get('_temperature')} | inputs={row.get('inputs')}")
        print(f"Most likely: {row.get('most_likely')}")
        print(f"Summary: {row.get('summary', '')[:400]}")
        print("\nDifferential:")
        for d in row["differential"]:
            print(
                f"  #{d['rank']}  {d['score']:>5}/100  {d['diagnosis']}  "
                f"[{d.get('confidence')}|{d.get('category')}]"
            )
        break
else:
    print("No differentials in results yet.")


Patient 10361982 HADM 24286431
Temp=0.4 | inputs={'symptom_tree': True, 'retained_snomed_entities': 4, 'retained_links': 4, 'clinical_context': True, 'information_extraction': True, 'prior_admissions': 1, 'prior_icd_codes': 11, 'prior_icd_context': True, 'prompt_format': 'ROLE/CONTEXT/TASK/CONSTRAINTS', 'temperature': 0.4}
Most likely: Post-hysterectomy hemorrhage
Summary: The patient presents with heavy vaginal bleeding after a hysterectomy, with a history of abnormal uterine bleeding and other gynecological issues.

Differential:
  #1   85.0/100  Post-hysterectomy hemorrhage  [high|primary]
  #2   70.0/100  Menorrhagia  [medium|secondary]
  #3   50.0/100  Postoperative adhesions  [medium|complication]
  #4   30.0/100  Hypothyroidism  [low|risk_factor]
